# 02 · Air temperature over the last minutes

A line chart of the air temperature of one zone. Each sensor of the tag
is its own line (`SENSOR` picks one). Change the settings below, then
**Run → Run All Cells**. Copy this notebook to your home folder before you change it (this
folder is read-only).

In [ ]:
# Settings: change these
UNIT = "UNIT01"        # a unit code, exactly as 01-connect lists it
ZONE = "ZONE01"        # a zone code of that unit
MINUTES = 60           # how far back to look
TAG = "air_temperature"
SENSOR = None          # None: every sensor of the tag, one line each; or e.g. "sht45_1"

In [ ]:
import matplotlib.pyplot as plt
import medcea_data as md

if md.whoami()["user"] is None:
    md.sign_in()

In [ ]:
df = md.measurements(UNIT, ZONE, tag=TAG, minutes=MINUTES, sensor=SENSOR)
print(f"{len(df)} readings")
df.tail()

In [ ]:
if df.empty:
    print(f"No {TAG} readings for {UNIT}/{ZONE} in the last {MINUTES} minutes. "
          "Check the codes with md.zones(), or look further back.")
else:
    uom = df["unit_of_measurement"].dropna().iloc[0] if df["unit_of_measurement"].notna().any() else ""
    fig, ax = plt.subplots(figsize=(10, 4))
    # one line per sensor: a board may carry two of one quantity (SHT45 and SCD30),
    # and "default" is a board flashed before sensor ids existed
    for name, part in df.groupby("sensor"):
        ax.plot(part["measure_timestamp"], part["value"], linewidth=1.5, label=name)
    if df["sensor"].nunique() > 1:
        ax.legend(title="sensor")
    span = f"latest {len(df)} readings" if df.attrs.get("truncated") else f"last {MINUTES} min"
    ax.set_title(f"{TAG.replace('_', ' ').capitalize()}, {UNIT} / {ZONE}, {span}")
    ax.set_xlabel("time (UTC)")
    ax.set_ylabel(f"{TAG.replace('_', ' ')} ({uom})" if uom else TAG.replace("_", " "))
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.show()